# Sanskrit model: resumable 1-billion-position GPU training

**Run this notebook in Google Colab:** open it, choose **Runtime → Run all**, allow Google Drive access, and select a GPU if Colab asks. The notebook downloads the saved model and the pinned repository automatically. You do not need to upload files manually.

The goal is **1,000,000,000 additional non-padding model-input positions**. This includes QA prompt tokens whose loss is masked; loss-bearing positions are reported separately. The small book/QA mix is repeatedly sampled, **not** one billion distinct tokens. More training can overfit or damage grammar; the material and model quality are not independently verified.

Checkpoints (model, optimizer and random states) are saved to `MyDrive/Sanskrit-LLM/colab-1b` approximately every five minutes with checksums. They are not automatically deleted, so Drive space can fill up. A disconnect, quota limit, runtime restart, crash, or full Drive stops training; rerun this notebook to resume from the last complete checkpoint. Work since the last save can be lost. Reaching the goal may require many sessions and may incur GPU charges. The notebook does **not** deploy or overwrite the live model. Do not run two copies against the same Drive folder.


In [ ]:
# One cell: GPU check, connect Drive, retrieve pinned code/model, train or resume.
import hashlib
import os
from pathlib import Path
import subprocess
import sys
import requests
import torch
from google.colab import drive

# These settings are ready to use; edit only if you intentionally want to change them.
TARGET_ADDITIONAL_POSITIONS = 1_000_000_000
BATCH_SIZE = 32
SAVE_EVERY_SECONDS = 300
LEARNING_RATE = 1e-5
REPO_COMMIT = "c7eaf212f1b69698e2e983f9aa608bf888a6e689"
RELEASE = "sanskrit-250m-44002147943"
ASSET = "book-qa-auto-44262147932-step-02000.pt"

if not torch.cuda.is_available():
    raise RuntimeError("GPU not found. In Colab choose Runtime > Change runtime type > GPU, then Run all again.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
drive.mount('/content/drive')
drive_dir = Path('/content/drive/MyDrive/Sanskrit-LLM/colab-1b')
drive_dir.mkdir(parents=True, exist_ok=True)

repo = Path('/content/HnS-sanskrit-1b')
if not (repo / '.git').exists():
    if repo.exists() and any(repo.iterdir()):
        raise RuntimeError(f"{repo} is not an empty Git repository; remove or rename it before retrying.")
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/Yuvii123-bot/HnS.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(repo), 'diff', '--exit-code'], check=True)

base_dir = Path('/content/sanskrit-checkpoints')
base_dir.mkdir(parents=True, exist_ok=True)
base = base_dir / ASSET
checksum = Path(str(base) + '.sha256')
url = f'https://github.com/Yuvii123-bot/HnS/releases/download/{RELEASE}/{ASSET}'

def download(address, target):
    temp = Path(str(target) + '.download')
    with requests.get(address, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with temp.open('wb') as out:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    out.write(chunk)
    os.replace(temp, target)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

download(url + '.sha256', checksum)
expected = checksum.read_text(encoding='utf-8').split()[0]
if not base.exists() or sha256(base) != expected:
    print('Downloading the saved checkpoint (about 119 MB)...', flush=True)
    download(url, base)
if sha256(base) != expected:
    raise RuntimeError('The downloaded checkpoint failed SHA-256 verification.')
print('Checkpoint verified. Starting or resuming training...', flush=True)
subprocess.run([
    sys.executable, '-u', 'scripts/colab_train_1b.py',
    '--base', str(base), '--drive-dir', str(drive_dir),
    '--target-tokens', str(TARGET_ADDITIONAL_POSITIONS),
    '--batch-size', str(BATCH_SIZE), '--save-seconds', str(SAVE_EVERY_SECONDS),
    '--lr', str(LEARNING_RATE),
], cwd=repo, check=True)
